## MLflow's Model Registry

In [4]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000/"

### Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI

In [7]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
print(client)
client.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/989310147491835230', creation_time=1751380259610, experiment_id='989310147491835230', last_update_time=1751380259610, lifecycle_stage='active', name='my-cool-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/496409895171607791', creation_time=1750830854497, experiment_id='496409895171607791', last_update_time=1750830854497, lifecycle_stage='active', name='green-taxi-duration', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/644981468207191929', creation_time=1748532732187, experiment_id='644981468207191929', last_update_time=1748532732187, lifecycle_stage='active', name='nyc-taxi-hw3', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/692710812252979065', creation_time=1748329845613, experiment_id='692710812252979065', last_update_time=1748329845613, lifecycle_stage='active', name='random-forest-hyperopt', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/339995047927440088', c

In [10]:
dir(client)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_check_artifact_file_string',
 '_create_model_version',
 '_get_registry_client',
 '_log_artifact_async_helper',
 '_log_artifact_helper',
 '_log_trace',
 '_raise_if_prompt',
 '_read_from_file',
 '_record_logged_model',
 '_registry_uri',
 '_start_tracked_trace',
 '_tracking_client',
 '_upload_ended_trace_info',
 '_upload_trace_data',
 '_validate_prompt',
 'copy_model_version',
 'create_experiment',
 'create_model_version',
 'create_registered_model',
 'create_run',
 'delete_assessment',
 'delete_experiment',
 'delete_model_version',
 'delete_model_version_tag',
 'delete_prompt',
 'delete_prompt_alias',
 'delete_registered_

In [11]:
client.get_experiment_by_name(name="random-forest-best-models")

<Experiment: artifact_location='mlflow-artifacts:/339995047927440088', creation_time=1748328436479, experiment_id='339995047927440088', last_update_time=1748328436479, lifecycle_stage='active', name='random-forest-best-models', tags={}>

Let's check the latest versions for the experiment with id `1`...

In [35]:
# Check if the experiment has run
runs_all = client.search_runs("339995047927440088")
print(f"Total runs found: {len(runs_all)}")

Total runs found: 6


In [36]:
runs = client.search_runs(
    experiment_ids="339995047927440088",
    max_results=5
)

for run in runs:
    print(f"run id: {run.info.run_id}, metrics: {run.data.metrics}")

run id: d2744aa915da45d581d06660ef7d6ec8, metrics: {'training_r2_score': 0.6796805248104354, 'test_rmse': 5.5941605655803635, 'training_score': 0.6796805248104354, 'val_rmse': 5.3633599989832135, 'training_mean_squared_error': 26.08294493276463, 'training_mean_absolute_error': 3.323916924052877, 'training_root_mean_squared_error': 5.107146456952711}
run id: d15c7bcf82e1480e8a8b53e42d3625b8, metrics: {'training_r2_score': 0.6773657330076874, 'test_rmse': 5.589460017934324, 'training_score': 0.6773657330076874, 'val_rmse': 5.357490752366866, 'training_mean_squared_error': 26.271433587992846, 'training_mean_absolute_error': 3.3210084471701498, 'training_root_mean_squared_error': 5.1255666601843}
run id: 0e91c85d9c1340b9bac8c7ffcba9b048, metrics: {'training_r2_score': 0.7027753287840568, 'test_rmse': 5.5921322796760755, 'training_score': 0.7027753287840568, 'val_rmse': 5.355041749098929, 'training_mean_squared_error': 24.20238334680275, 'training_mean_absolute_error': 3.2487582305772666, '

In [37]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='339995047927440088',
    filter_string="metrics.val_rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [41]:
for run in runs:
    print(f"run id: {run.info.run_id}, val_rmse: {run.data.metrics['val_rmse']:.4f}")

run id: d2744aa915da45d581d06660ef7d6ec8, val_rmse: 5.3634
run id: d15c7bcf82e1480e8a8b53e42d3625b8, val_rmse: 5.3575
run id: 0e91c85d9c1340b9bac8c7ffcba9b048, val_rmse: 5.3550
run id: ba38f1edf5cf47f98e7938a588b26b3b, val_rmse: 5.3547
run id: 5fec554eefc74fac9beff9ad49ee42f4, val_rmse: 5.3354


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `4` was created.
3. Transition the version `4` to "Staging" and adding annotations to it.

In [40]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [42]:
run_id = "d2744aa915da45d581d06660ef7d6ec8"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="random-forest-regressor-best")

Registered model 'random-forest-regressor-best' already exists. Creating a new version of this model...
2025/07/01 19:59:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random-forest-regressor-best, version 3
Created version '3' of model 'random-forest-regressor-best'.


<ModelVersion: aliases=[], creation_timestamp=1751381942301, current_stage='None', description='', last_updated_timestamp=1751381942301, name='random-forest-regressor-best', run_id='d2744aa915da45d581d06660ef7d6ec8', run_link='', source='mlflow-artifacts:/339995047927440088/d2744aa915da45d581d06660ef7d6ec8/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='3'>

In [43]:
model_name = "random-forest-regressor-best"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 2, stage: Production
version: 1, stage: Staging
version: 3, stage: None


/tmp/ipykernel_230372/1216751292.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [44]:
model_version = 3
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_230372/1957332551.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1751381942301, current_stage='Staging', description='', last_updated_timestamp=1751382060266, name='random-forest-regressor-best', run_id='d2744aa915da45d581d06660ef7d6ec8', run_link='', source='mlflow-artifacts:/339995047927440088/d2744aa915da45d581d06660ef7d6ec8/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='3'>

In [45]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1751381942301, current_stage='Staging', description='The model version 3 was transitioned to Staging on 2025-07-01', last_updated_timestamp=1751382078947, name='random-forest-regressor-best', run_id='d2744aa915da45d581d06660ef7d6ec8', run_link='', source='mlflow-artifacts:/339995047927440088/d2744aa915da45d581d06660ef7d6ec8/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='3'>

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [48]:
from sklearn.metrics import mean_squared_error
import pandas as pd


def read_dataframe(filename):
    if filename.endswith(".csv"):
        df = pd.read_csv(filename)
    else:
        df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": mean_squared_error(y_test, y_pred, squared=False)}

In [49]:
df = read_dataframe("./../data/green_tripdata_2021-02.parquet")

In [64]:
"""muhammad extra block of code...
# unfortunately I had not added the dv.pkl(preprocessor) to run in the experiment, but that;s great it was saved
# locally in my system/repo. Now adding it with all runs, some imports are already defined""" 

# It's already defined above
# from mlflow.tracking import MlflowClient

# client = MlflowClient()
experiment_id = "339995047927440088"
artifact_file = "./../homework/processed_output_data/dv.pkl"

# Get all runs for the experiment
runs = client.search_runs(experiment_id)

for run in runs:
    run_id = run.info.run_id
    print(f"Logging artifact to run: {run_id}")
    client.log_artifact(run_id, artifact_file, artifact_path="preprocessor")


Logging artifact to run: d2744aa915da45d581d06660ef7d6ec8
Logging artifact to run: d15c7bcf82e1480e8a8b53e42d3625b8
Logging artifact to run: 0e91c85d9c1340b9bac8c7ffcba9b048
Logging artifact to run: ba38f1edf5cf47f98e7938a588b26b3b
Logging artifact to run: 5fec554eefc74fac9beff9ad49ee42f4
Logging artifact to run: 685b2e62f1814ad1a5d6073b69221140


In [65]:
# I am going to download the preprocessor now, previously it was not there that's why done block of code
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/notebooks/preprocessor'

In [66]:
import pickle

with open("preprocessor/dv.pkl", "rb") as f_in:
    dv = pickle.load(f_in)

In [67]:
X_test = preprocess(df, dv)

In [68]:
target = "duration"
y_test = df[target].values

In [72]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

2025/07/01 20:43:53 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - psutil (current: 5.9.8, required: psutil==5.9.4)
 - scikit-learn (current: 1.0.2, required: scikit-learn==1.2.1)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


CPU times: user 397 ms, sys: 95.3 ms, total: 493 ms
Wall time: 561 ms


/home/mshifa/Downloads/ye/envs/main/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.2.1 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mshifa/Downloads/ye/envs/main/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator RandomForestRegressor from version 1.2.1 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(


{'rmse': 6.678125236694914}

In [75]:
# mlflow always take the latest models if there are multiple models in the same stage

%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

2025/07/01 20:49:49 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - psutil (current: 5.9.8, required: psutil==5.9.4)
 - scikit-learn (current: 1.0.2, required: scikit-learn==1.2.1)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
/home/mshifa/Downloads/ye/envs/main/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.2.1 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mshifa/Downloads/ye/envs/main/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator Rand

CPU times: user 380 ms, sys: 39.1 ms, total: 419 ms
Wall time: 445 ms


{'rmse': 6.66887679442565}

In [76]:
# transition the model version stages, version 3 in staging and version 1 in production
# changing version 3 to production

client.transition_model_version_stage(
    name=model_name,
    version=3,
    stage="Production",
    archive_existing_versions=True
)

# after running this block, version 3 successfully in production while version 1 is in Archive

/tmp/ipykernel_230372/914890349.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1751381942301, current_stage='Production', description='The model version 3 was transitioned to Staging on 2025-07-01', last_updated_timestamp=1751385119835, name='random-forest-regressor-best', run_id='d2744aa915da45d581d06660ef7d6ec8', run_link='', source='mlflow-artifacts:/339995047927440088/d2744aa915da45d581d06660ef7d6ec8/artifacts/model', status='READY', status_message=None, tags={}, user_id='', version='3'>